# Hacking Forums — 04: Síntesis e informe

Este es el notebook final del caso. No corre ningún modelo nuevo — lo que hace es **convergir**  
todas las señales de los notebooks anteriores y responder las preguntas de investigación  
planteadas en el notebook 00.

En investigación forense, el informe final no es un resumen automático:  
es la interpretación humana de los hallazgos. Los modelos detectan patrones;  
el analista decide cuáles son significativos y cuáles son ruido.

### Preguntas de investigación (del notebook 00)

1. ¿Hay usuarios que aparecen en múltiples foros?
2. ¿Cada foro tiene una especialización temática detectable?
3. ¿Los actores cross-foro tienen patrones de comportamiento consistentes?

**Prerequisito**: haber ejecutado los notebooks 01, 02 y 03 para generar todos los artefactos.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go

from src.utils import RESULTS_DIR

plt.style.use('dark_background')
sns.set_palette('muted')

HF_RESULTS = RESULTS_DIR / 'hacking_forums'

# Cargar todos los artefactos disponibles
def safe_load(path):
    """Carga un parquet si existe, devuelve DataFrame vacío si no."""
    if path.exists():
        return pd.read_parquet(path)
    print(f"  [NO ENCONTRADO] {path.name}")
    return pd.DataFrame()

posts_clean        = safe_load(HF_RESULTS / 'posts_clean.parquet')
users_clean        = safe_load(HF_RESULTS / 'users_clean.parquet')
ner_df             = safe_load(HF_RESULTS / 'ner_hf_results.parquet')
profiles_df        = safe_load(HF_RESULTS / 'actor_profiles.parquet')
candidates_df      = safe_load(HF_RESULTS / 'pivoting_candidates_enriched.parquet')

if candidates_df.empty:
    candidates_df  = safe_load(HF_RESULTS / 'pivoting_candidates.parquet')

print("\n=== Artefactos cargados ===")
print(f"posts_clean:         {len(posts_clean):,} filas")
print(f"users_clean:         {len(users_clean):,} filas")
print(f"ner_df:              {len(ner_df):,} entidades")
print(f"profiles_df:         {len(profiles_df):,} perfiles")
print(f"candidates_df:       {len(candidates_df):,} candidatos")

## Pregunta 1 — ¿Hay usuarios en múltiples foros?

Esta pregunta tiene tres capas de evidencia, cada una más robusta que la anterior:

| Señal | Descripción | Fortaleza |
|-------|-------------|----------|
| Handle exacto | Mismo username case-insensitive | Necesaria pero no suficiente |
| Handle fuzzy ≥85% | Variaciones del mismo handle | Evidencia suplementaria |
| Embedding similar | Mismo estilo semántico cross-foro | Confirmación independiente |

La **convergencia** de las tres señales en el mismo par de usuarios  
es lo que transforma un candidato en una atribución sólida.

**Un handle compartido por sí solo NO es suficiente** — nombres como `admin`, `user123`  
o `shadow` son extremadamente comunes y aparecerán en todos los foros sin ser la misma persona.

In [ ]:
if not candidates_df.empty:
    # Candidatos con handle exacto
    exact_mask = candidates_df.get('match_type', pd.Series(dtype=str)) == 'exact'
    fuzzy_mask = candidates_df.get('match_type', pd.Series(dtype=str)) == 'fuzzy'

    n_exact = exact_mask.sum() if 'match_type' in candidates_df.columns else 0
    n_fuzzy = fuzzy_mask.sum() if 'match_type' in candidates_df.columns else 0

    print(f"=== Pivoting de handles ===")
    print(f"Candidatos con handle exacto:  {n_exact:,}")
    print(f"Candidatos fuzzy ≥85%:         {n_fuzzy:,}")

    # Si hay embedding_sim, mostrar candidatos con múltiples señales
    if 'embedding_sim' in candidates_df.columns:
        high_conf = candidates_df[
            (candidates_df['handle_sim'].fillna(0) >= 0.85) &
            (candidates_df['embedding_sim'].fillna(0) >= 0.70)
        ].copy()
        print(f"\nCandidatos con handle ≥85% Y embedding ≥0.70: {len(high_conf):,}")
        print("(Son los candidatos más sólidos de atribución cross-foro)")
        if not high_conf.empty:
            cols = ['handle_a', 'forum_a', 'handle_b', 'forum_b', 'handle_sim', 'embedding_sim']
            cols = [c for c in cols if c in high_conf.columns]
            print(high_conf[cols].head(20).to_string(index=False))
    else:
        print(f"\nTop 20 candidatos (embedding_sim no disponible — ejecutar notebook 03):")
        cols = [c for c in ['handle_a', 'forum_a', 'handle_b', 'forum_b', 'handle_sim'] if c in candidates_df.columns]
        print(candidates_df[cols].head(20).to_string(index=False))
else:
    print("Sin candidatos de pivoting — ejecutar notebooks 02 y 03 primero.")

## Tabla de convergencia de señales

Esta es la tabla central del informe forense.  
Para cada candidato de atribución, mostramos todas las señales disponibles:

- `handle_sim`: similitud del handle (1.0 = exacto, 0.85+ = fuzzy)
- `embedding_sim`: similitud semántica (0.70+ = alta)
- `role_a` / `role_b`: rol inferido por el modelo en cada foro
- `n_signals`: cuántas señales independientes convergieron

Un candidato con **3 señales convergentes** es candidato de alta confianza.

In [ ]:
if not candidates_df.empty:
    convergence = candidates_df.copy()

    # Calcular score de señales convergentes
    convergence['sig_handle']    = (convergence.get('handle_sim', 0).fillna(0) >= 0.85).astype(int)
    convergence['sig_embedding'] = (convergence.get('embedding_sim', 0).fillna(0) >= 0.70).astype(int)
    convergence['n_signals']     = convergence['sig_handle'] + convergence['sig_embedding']

    # Si hay perfiles disponibles, agregar rol de cada usuario
    if not profiles_df.empty and 'role' in profiles_df.columns:
        profile_map = {f"{row['forum']}::{row['userid']}": row['role']
                       for _, row in profiles_df.iterrows()
                       if 'forum' in profiles_df.columns and 'userid' in profiles_df.columns}

    # Ordenar por número de señales y embedding_sim
    sort_cols = ['n_signals']
    if 'embedding_sim' in convergence.columns:
        sort_cols.append('embedding_sim')
    convergence = convergence.sort_values(sort_cols, ascending=False)

    print("=== Tabla de convergencia de señales ===")
    print(f"Candidatos con ≥2 señales: {(convergence['n_signals'] >= 2).sum():,}")
    print(f"Candidatos con 1 señal:    {(convergence['n_signals'] == 1).sum():,}")
    print()

    display_cols = [c for c in ['handle_a', 'forum_a', 'handle_b', 'forum_b',
                                 'handle_sim', 'embedding_sim', 'match_type', 'n_signals']
                    if c in convergence.columns]
    print(convergence[display_cols].head(30).to_string(index=False))

    # Guardar tabla de convergencia
    convergence.to_parquet(HF_RESULTS / 'convergence_table.parquet', index=False)
    print(f"\nGuardado: {HF_RESULTS / 'convergence_table.parquet'}")
else:
    print("Sin candidatos para tabla de convergencia.")

## Pregunta 2 — ¿Especialización temática por foro?

Del notebook 02 obtuvimos los términos TF-IDF más representativos de cada foro.  
Aquí construimos la interpretación final.

Si el TF-IDF de RaidForums tiene `leak`, `data`, `database` entre sus top términos  
y OGUsers tiene `og`, `username`, `accounts`, podemos afirmar que:
- OGUsers estaba especializado en cuentas de usuarios premium
- RaidForums en leaks de bases de datos

Eso es **inteligencia** extraída del corpus, no solo estadística.

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer

if len(posts_clean) > 0:
    # Reconstruir corpus TF-IDF por foro
    forum_corpus = {}
    for forum, grp in posts_clean.groupby('forum'):
        combined = ' '.join(grp['pagetext'].dropna().astype(str).tolist())
        combined = re.sub(r'<[^>]+>', ' ', combined)
        combined = re.sub(r'[^a-zA-Z ]', ' ', combined)
        forum_corpus[forum] = combined

    forums_ordered = sorted(forum_corpus.keys())
    corpus_docs    = [forum_corpus[f] for f in forums_ordered]

    vectorizer = TfidfVectorizer(
        max_features=5000, min_df=2, stop_words='english', ngram_range=(1, 2)
    )
    tfidf_matrix = vectorizer.fit_transform(corpus_docs)
    feature_names = vectorizer.get_feature_names_out()
    tfidf_dense   = tfidf_matrix.toarray()

    print("=== Especialización temática por foro (top 15 términos TF-IDF) ===\n")
    specialization = {}
    for idx, forum in enumerate(forums_ordered):
        scores     = tfidf_dense[idx]
        top_idx    = scores.argsort()[-15:][::-1]
        top_terms  = feature_names[top_idx].tolist()
        specialization[forum] = top_terms
        print(f"  {forum.split('_')[0]:<20} → {', '.join(top_terms[:10])}")

    print("\n=== Interpretación (ejemplo — completar con análisis real) ===")
    print("  OGUsers        → términos relacionados con cuentas OG y usernames premium")
    print("  Exploit.in     → términos de exploits, vulnerabilidades, discusión técnica en ruso")
    print("  Cracked.to     → cracking de cuentas, combo lists, proxies")
    print("  Nulled.io      → nulled scripts, herramientas crakeadas, hosting")
    print("  RaidForums     → leaks de bases de datos, doxing, raids")
else:
    print("Sin posts para análisis TF-IDF.")

## Pregunta 3 — ¿Patrones de comportamiento cross-foro?

Para los actores que aparecen en múltiples foros, preguntamos:  
¿actúan de la misma manera en todos? ¿o cambian de rol según el foro?

Esto es lo más difícil de detectar y lo más valioso forensicamente.  
Un actor que **vende** en un foro y **compra** en otro revela una cadena de valor.

Combinamos:
- Perfiles de actores del notebook 03 (rol inferido por el modelo)
- Candidatos de pivoting con alta confianza
- Entidades NER del notebook 03 (qué herramientas/malware mencionan)

In [ ]:
if not candidates_df.empty and not profiles_df.empty:
    # Identificar candidatos de alta confianza (con embedding_sim si disponible)
    if 'embedding_sim' in candidates_df.columns:
        top_candidates = candidates_df[
            candidates_df['embedding_sim'].fillna(0) >= 0.65
        ].copy()
    else:
        top_candidates = candidates_df[candidates_df.get('handle_sim', 0).fillna(0) >= 0.90].copy()

    print(f"Candidatos de alta confianza: {len(top_candidates):,}")

    if not top_candidates.empty and 'role' in profiles_df.columns:
        # Intentar enlazar con perfiles
        profile_map_by_user = {}
        if 'username' in profiles_df.columns and 'forum' in profiles_df.columns:
            for _, p_row in profiles_df.iterrows():
                key = (p_row.get('username', ''), p_row['forum'])
                profile_map_by_user[key] = {
                    'role': p_row.get('role', ''),
                    'confidence': p_row.get('confidence', ''),
                    'rationale': p_row.get('rationale', ''),
                }

        print("\n=== Actores cross-foro con perfil de rol ===")
        for _, pair in top_candidates.head(10).iterrows():
            handle_a = pair.get('handle_a', '')
            handle_b = pair.get('handle_b', '')
            forum_a  = pair.get('forum_a', '')
            forum_b  = pair.get('forum_b', '')
            emb_sim  = pair.get('embedding_sim', None)
            h_sim    = pair.get('handle_sim', None)

            profile_a = profile_map_by_user.get((handle_a, forum_a), {})
            profile_b = profile_map_by_user.get((handle_b, forum_b), {})

            print(f"\n  {handle_a} ({forum_a.split('_')[0]}) ↔ {handle_b} ({forum_b.split('_')[0]})")
            if h_sim is not None:
                print(f"    Handle sim:    {h_sim:.3f}")
            if emb_sim is not None:
                print(f"    Embedding sim: {emb_sim:.3f}")
            if profile_a:
                print(f"    Rol en {forum_a.split('_')[0]}: {profile_a.get('role', 'N/A')} ({profile_a.get('confidence', '')})")
                print(f"    → {profile_a.get('rationale', '')}")
            if profile_b:
                print(f"    Rol en {forum_b.split('_')[0]}: {profile_b.get('role', 'N/A')} ({profile_b.get('confidence', '')})")
                print(f"    → {profile_b.get('rationale', '')}")
    else:
        print("Perfiles no disponibles — ejecutar notebook 03 para agregar roles.")
elif candidates_df.empty:
    print("Sin candidatos — ejecutar notebook 02 para generarlos.")
elif profiles_df.empty:
    print("Sin perfiles — ejecutar notebook 03 para generarlos.")

## Visualización: distribución de NER cross-foro

¿Qué herramientas y malware mencionan los actores más activos?  
Esta visualización consolida los hallazgos de NER del notebook 03  
para los foros en inglés.

In [ ]:
if not ner_df.empty and 'entity_type' in ner_df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    entity_types = ['TOOL', 'MALWARE', 'CVE', 'MARKETPLACE']
    colors = ['#4E9EE9', '#E94E4E', '#E9A24E', '#4EE97A']

    for idx, (etype, color) in enumerate(zip(entity_types, colors)):
        sub = ner_df[ner_df['entity_type'] == etype]
        if sub.empty:
            axes[idx].text(0.5, 0.5, f'Sin datos {etype}', ha='center', transform=axes[idx].transAxes)
            axes[idx].set_title(etype)
            continue

        top_ents = sub['entity'].str.lower().value_counts().head(15)
        axes[idx].barh(top_ents.index[::-1], top_ents.values[::-1], color=color, alpha=0.85)
        axes[idx].set_title(f'Top 15 {etype}')
        axes[idx].set_xlabel('Menciones')
        for i, v in enumerate(top_ents.values[::-1]):
            axes[idx].text(v + 0.2, i, str(v), va='center', fontsize=8)

    plt.suptitle('Entidades NER más frecuentes — foros underground en inglés', fontsize=13)
    plt.tight_layout()
    plt.savefig(HF_RESULTS / 'sintesis_ner_crossforum.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("NER no disponible — ejecutar notebook 03 primero.")

## Conclusiones del análisis

Esta sección debe completarse con los hallazgos reales del análisis ejecutado.  
Las conclusiones que siguen son una **plantilla** — reemplazarla con los resultados reales.

### Sobre la presencia cross-foro de actores

> *(Completar con los N candidatos de alta confianza identificados, sus handles,  
> foros donde aparecen, y el score de convergencia de señales.)*

### Sobre la especialización temática

> *(Completar con los términos TF-IDF dominantes de cada foro y la interpretación  
> de qué tipo de actividad underground caracteriza a cada uno.)*

### Sobre los patrones de comportamiento

> *(Completar con los perfiles de actores cross-foro: ¿mantienen el mismo rol?  
> ¿Hay cambios de rol que sugieran evolución en el ecosistema?)*

---

## Limitaciones del análisis

1. **Temporal**: Los dumps tienen fechas distintas (2013–2021). Los actores pueden haber  
   cambiado de identidad entre dumps de distintas épocas.

2. **Exploit.in excluido del NER**: Por las razones metodológicas documentadas en el notebook 03,  
   los actores de Exploit.in solo participan del análisis de embeddings y clustering,  
   no del NER ni del perfilado.

3. **Fuzzy matching con cap**: El matching fuzzy se limitó a 500 handles por foro por performance.  
   Un análisis exhaustivo requeriría más tiempo de cómputo.

4. **Handles comunes**: Handles genéricos (`admin`, `user`, `shadow`) pueden generar  
   falsos positivos en el matching. La similitud de embeddings es la señal que descarta  
   estas coincidencias accidentales.

5. **Perfilado de roles**: La inferencia de roles con qwen2.5:14b es una estimación  
   probabilística. Las categorías no son mutuamente excluyentes y requieren validación manual.

## Exportación del informe

Exportamos los resultados finales en formatos que pueden compartirse con el equipo:  
- `convergence_table.csv`: tabla de candidatos con todas las señales (legible en Excel)
- `actor_profiles.csv`: perfiles de actores con roles inferidos
- Las visualizaciones ya se guardaron como PNG en las celdas anteriores

**Para exportar el informe completo como HTML** (incluyendo las visualizaciones Plotly interactivas):  
```bash
jupyter nbconvert --to html 04_sintesis_informe.ipynb --output informe_final.html
```

In [ ]:
export_paths = []

# Exportar tabla de convergencia
conv_parquet = HF_RESULTS / 'convergence_table.parquet'
if conv_parquet.exists():
    conv_df = pd.read_parquet(conv_parquet)
    csv_path = HF_RESULTS / 'convergence_table.csv'
    conv_df.to_csv(csv_path, index=False)
    export_paths.append(csv_path)
    print(f"✓  convergence_table.csv ({len(conv_df):,} candidatos)")

# Exportar perfiles de actores
if not profiles_df.empty:
    profiles_csv = HF_RESULTS / 'actor_profiles.csv'
    profiles_df.to_csv(profiles_csv, index=False)
    export_paths.append(profiles_csv)
    print(f"✓  actor_profiles.csv ({len(profiles_df):,} perfiles)")

# Exportar estadísticas del corpus
if len(posts_clean) > 0 and len(users_clean) > 0:
    stats_rows = []
    for forum in sorted(posts_clean['forum'].unique()):
        n_posts = (posts_clean['forum'] == forum).sum()
        n_users = (users_clean['forum'] == forum).sum() if 'forum' in users_clean.columns else 0
        stats_rows.append({'forum': forum, 'posts': n_posts, 'users': n_users})
    stats_df = pd.DataFrame(stats_rows)
    stats_csv = HF_RESULTS / 'corpus_stats.csv'
    stats_df.to_csv(stats_csv, index=False)
    export_paths.append(stats_csv)
    print(f"✓  corpus_stats.csv")
    print()
    print(stats_df.to_string(index=False))

print(f"\nArchivos exportados a: {HF_RESULTS}")
for p in export_paths:
    size = p.stat().st_size / 1024
    print(f"  {p.name} ({size:.1f} KB)")

## Resumen ejecutivo

Este notebook cierra el ciclo analítico del caso HackingForums.  
Lo que logramos en los 5 notebooks:

| Notebook | Trabajo realizado |
|----------|------------------|
| 00 Reconocimiento | Evaluación de 7 dumps, selección de 5, detección de idioma por foro |
| 01 Ingeniería de datos | Carga, limpieza, normalización, corpus por usuario |
| 02 Análisis estructural | Grafo de co-participación, pivoting exacto/fuzzy, TF-IDF |
| 03 Análisis semántico | Embeddings, UMAP/HDBSCAN, BERTopic, NER, perfilado de actores |
| 04 Síntesis | Convergencia de señales, conclusiones, exportación |

### ¿Qué sería el siguiente paso en un caso real?

1. **Validación manual** de los top-N candidatos de atribución con mayor n_signals
2. **Correlación con otras fuentes**: telegram dumps, clearweb profiles, OSINT
3. **Análisis de redes ampliado**: grafo multi-foro con los candidatos confirmados como nodos especiales
4. **NER en ruso** para Exploit.in con modelos apropiados (DeepPavlov/rubert o spacy ru_core_news_lg)
5. **Análisis temporal** de los actores: ¿en qué período estuvieron más activos?

---

*Análisis generado con el pipeline del curso CSBC26 — Caso 2: Hacking Forums*